#MapReduce
MapReduce is a distributed data processing framework that stores intermediate results on disk after every processing stage

Limitations of MapReduce are:-
1. Slower performance for iterative algorithms and real-time processing
2. Not suitable for interactive data analysis
3. Increased execution time for large-scale data processing tasks
4. High disk I/O overhead due to frequent read/write operations

# Apache Spark
Apache Spark is a distributed computing framework designed to overcome the limitations of MapReduce.

##Advantages of Apache Spark:-
1. In-Memory Processing: Stores intermediate data in memory, reducing disk access.
2. High Performance: Can be significantly faster than MapReduce for many workloads.
3. Easy APIs: Supports Python, Scala, Java, and R.
4. Fault Tolerance: Uses lineage information to recover lost data.
5. Scalability: Can process very large datasets across multiple machines.

#Spark DataFrames and Immutability
A DataFrame is a distributed collection of data organized into named columns, similar to a table in a relational database.

###Characteristics of DataFrames
1. Structured and schema-based.
2. Optimized using Spark's Catalyst Optimizer.
3. Support SQL-like operations.
4. Efficient for large-scale analytics.

###Immutability
Spark DataFrames are immutable, meaning they cannot be modified directly.

Example:

df2 = df.filter(df.age > 25)

The original DataFrame df remains unchanged. Spark creates a new DataFrame df2 containing the filtered results.

In [4]:
from google.colab import files

uploaded = files.upload()

Saving customer_orders.csv to customer_orders.csv


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week_5_Assignment") \
    .getOrCreate()

In [17]:
df = spark.read.csv(
    "customer_orders.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-----------+-------------+---+------+----------+-------+-----------+------------+--------+-----+-----------+----------+------+
|customer_id|customer_name|age|gender|      city| region|   category|product_name|quantity|price|total_sales|order_date|rating|
+-----------+-------------+---+------+----------+-------+-----------+------------+--------+-----+-----------+----------+------+
|        100|        Sneha| 20|Female|   Lucknow|  South|   Clothing|       Kurti|       2|35000|      70000|12/18/2026|     1|
|        101|         Yash| 46|  Male|     Delhi|  North|   Clothing|       Jeans|       9|18000|     162000| 1/18/2026|     2|
|        102|        Meera| 45|  Male|   Chennai|Central|  Furniture|   Bookshelf|       1|  500|        500|12/14/2026|     3|
|        103|         NULL| 28|  Male|Chandigarh|  North|Electronics|          TV|       2| 2000|       4000| 6/20/2026|     3|
|        104|        Priya| 48|  Male|   Kolkata|  North|     Sports|  Basketball|      10| 2000|      2

In [7]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- total_sales: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- rating: integer (nullable = true)



In [18]:
print("Total Rows:", df.count())

Total Rows: 105


In [9]:
df.groupBy("customer_id") \
  .count() \
  .filter("count > 1") \
  .show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|        140|    2|
|        112|    2|
|        160|    2|
|        125|    2|
|        105|    2|
+-----------+-----+



In [21]:
print("Before:", df.count())

df = df.dropDuplicates()

print("After:", df.count())

Before: 105
After: 100


In [11]:
df.groupBy("customer_id") \
  .count() \
  .filter("count > 1") \
  .show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [19]:
from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----------+-------------+---+------+----+------+--------+------------+--------+-----+-----------+----------+------+
|customer_id|customer_name|age|gender|city|region|category|product_name|quantity|price|total_sales|order_date|rating|
+-----------+-------------+---+------+----+------+--------+------------+--------+-----+-----------+----------+------+
|          0|            3|  1|     0|   0|    15|       0|           0|       0|    0|          0|         0|     0|
+-----------+-------------+---+------+----+------+--------+------------+--------+-----+-----------+----------+------+



In [25]:
df = df.fillna({
    "age": 0,
    "region": "Unknown"
})
df.orderBy("customer_id").show(10)

+-----------+-------------+---+------+----------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|customer_id|customer_name|age|gender|      city| region|   category| product_name|quantity|price|total_sales|order_date|rating|
+-----------+-------------+---+------+----------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|        100|        Sneha| 20|Female|   Lucknow|  South|   Clothing|        Kurti|       2|35000|      70000|12/18/2026|     1|
|        101|         Yash| 46|  Male|     Delhi|  North|   Clothing|        Jeans|       9|18000|     162000| 1/18/2026|     2|
|        102|        Meera| 45|  Male|   Chennai|Central|  Furniture|    Bookshelf|       1|  500|        500|12/14/2026|     3|
|        103|         NULL| 28|  Male|Chandigarh|  North|Electronics|           TV|       2| 2000|       4000| 6/20/2026|     3|
|        104|        Priya| 48|  Male|   Kolkata|  North|     Sports|   Basketball|      10| 2000

In [27]:
df.filter(col("customer_name").isNull()).show()

+-----------+-------------+---+------+----------+-------+-----------+------------+--------+-----+-----------+----------+------+
|customer_id|customer_name|age|gender|      city| region|   category|product_name|quantity|price|total_sales|order_date|rating|
+-----------+-------------+---+------+----------+-------+-----------+------------+--------+-----+-----------+----------+------+
|        155|         NULL| 48|  Male| Ahmedabad|Central|Electronics|          TV|      10|18000|     180000| 11/1/2026|     1|
|        118|         NULL| 33|  Male|     Delhi|  North|Electronics|      Mobile|       2|  100|        200|  6/3/2026|     5|
|        103|         NULL| 28|  Male|Chandigarh|  North|Electronics|          TV|       2| 2000|       4000| 6/20/2026|     3|
+-----------+-------------+---+------+----------+-------+-----------+------------+--------+-----+-----------+----------+------+



In [29]:
from pyspark.sql.functions import *

df = df.filter(
    col("customer_name").isNotNull() &
    (trim(col("customer_name")) != "")
)
df.orderBy("customer_id").show(10)

+-----------+-------------+---+------+-------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|customer_id|customer_name|age|gender|   city| region|   category| product_name|quantity|price|total_sales|order_date|rating|
+-----------+-------------+---+------+-------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|        100|        Sneha| 20|Female|Lucknow|  South|   Clothing|        Kurti|       2|35000|      70000|12/18/2026|     1|
|        101|         Yash| 46|  Male|  Delhi|  North|   Clothing|        Jeans|       9|18000|     162000| 1/18/2026|     2|
|        102|        Meera| 45|  Male|Chennai|Central|  Furniture|    Bookshelf|       1|  500|        500|12/14/2026|     3|
|        104|        Priya| 48|  Male|Kolkata|  North|     Sports|   Basketball|      10| 2000|      20000| 10/7/2026|     1|
|        105|        Priya| 33|Female| Mumbai|  South|Electronics|           TV|       5| 5000|      25000|11/27/2026|

In [30]:
df = df.withColumnRenamed(
    "customer_name",
    "customer_full_name"
)
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_full_name: string (nullable = true)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = false)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- total_sales: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- rating: integer (nullable = true)



In [31]:
from pyspark.sql.functions import col

df = df.withColumn(
    "age",
    col("age").cast("int")
)
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_full_name: string (nullable = true)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = false)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- total_sales: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- rating: integer (nullable = true)



In [32]:
df.filter(
    (col("age") >= 20) &
    (col("age") <= 40)
).show()

+-----------+------------------+---+------+---------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|customer_id|customer_full_name|age|gender|     city| region|   category| product_name|quantity|price|total_sales|order_date|rating|
+-----------+------------------+---+------+---------+-------+-----------+-------------+--------+-----+-----------+----------+------+
|        130|             Karan| 27|Female|   Mumbai|  South|  Furniture|         Sofa|       3| 5000|      15000| 9/23/2026|     3|
|        167|              Neha| 38|Female|  Chennai|Central|     Sports|   Basketball|       7|55000|     385000| 9/11/2026|     3|
|        109|              Aman| 28|Female|   Jaipur|  South|     Sports|Tennis Racket|       5|55000|     275000|10/14/2026|     5|
|        116|             Meera| 29|Female|Hyderabad|Central|    Grocery|          Oil|       9|55000|     495000| 12/7/2026|     3|
|        105|             Priya| 33|Female|   Mumbai|  South|Electron

In [33]:
df.filter(
    col("category") == "Electronics"
).show()

+-----------+------------------+---+------+---------+-------+-----------+------------+--------+-----+-----------+----------+------+
|customer_id|customer_full_name|age|gender|     city| region|   category|product_name|quantity|price|total_sales|order_date|rating|
+-----------+------------------+---+------+---------+-------+-----------+------------+--------+-----+-----------+----------+------+
|        139|            Simran| 51|Female|Hyderabad|   East|Electronics|      Laptop|       5|  500|       2500| 10/9/2026|     1|
|        105|             Priya| 33|Female|   Mumbai|  South|Electronics|          TV|       5| 5000|      25000|11/27/2026|     3|
|        173|             Vikas| 28|Female|   Mumbai|  South|Electronics|  Headphones|       7|18000|     126000|10/26/2026|     5|
|        170|             Pooja| 33|Female|   Jaipur|   East|Electronics|     Monitor|       9|  500|       4500|  5/2/2026|     1|
|        117|            Manish| 42|Female|Hyderabad|   West|Electronics|   

In [34]:
df.filter(
    col("region") == "North"
).show()

+-----------+------------------+---+------+----------+------+-----------+------------+--------+-----+-----------+----------+------+
|customer_id|customer_full_name|age|gender|      city|region|   category|product_name|quantity|price|total_sales|order_date|rating|
+-----------+------------------+---+------+----------+------+-----------+------------+--------+-----+-----------+----------+------+
|        101|              Yash| 46|  Male|     Delhi| North|   Clothing|       Jeans|       9|18000|     162000| 1/18/2026|     2|
|        104|             Priya| 48|  Male|   Kolkata| North|     Sports|  Basketball|      10| 2000|      20000| 10/7/2026|     1|
|        156|             Kavya| 27|Female|    Jaipur| North|  Furniture|    Cupboard|       6|  800|       4800| 8/11/2026|     3|
|        158|             Priya| 20|  Male|   Lucknow| North|     Sports|    Football|       4|  500|       2000| 8/22/2026|     1|
|        171|             Meera| 37|  Male|   Chennai| North|Electronics|  H

In [35]:
from pyspark.sql.functions import *

df.select(
    count("*").alias("total_records"),
    sum("total_sales").alias("total_sales"),
    avg("total_sales").alias("avg_sales"),
    min("total_sales").alias("min_sales"),
    max("total_sales").alias("max_sales")
).show()

+-------------+-----------+-----------------+---------+---------+
|total_records|total_sales|        avg_sales|min_sales|max_sales|
+-------------+-----------+-----------------+---------+---------+
|           97|    6778600|69882.47422680413|      250|   550000|
+-------------+-----------+-----------------+---------+---------+



#Wide Transformations and Shuffle Operations
Spark transformations are classified as narrow and wide transformations. In a wide transformation, data from multiple partitions is required to perform the operation. As a result, Spark redistributes data across partitions through a process called shuffle.

Operations such as groupBy(), join(), and distinct() are examples of wide transformations.
###For example:

df.groupBy("region").count().show()

In this operation, Spark groups records with the same region together, which requires moving data between partitions. Although wide transformations are useful for aggregation and analysis, shuffle operations increase execution time because they involve network communication and additional processing.

In [36]:
df.groupBy("region") \
  .agg(
      sum("total_sales").alias("total_sales")
  ) \
  .show()

+-------+-----------+
| region|total_sales|
+-------+-----------+
|Unknown|     396250|
|  South|    1226800|
|Central|    1747850|
|   East|    1609550|
|   West|     697800|
|  North|    1100350|
+-------+-----------+



In [37]:
df.groupBy("category") \
  .agg(
      avg("total_sales").alias("avg_sales")
  ) \
  .show()

+-----------+------------------+
|   category|         avg_sales|
+-----------+------------------+
|     Sports| 65068.42105263158|
|    Grocery|104417.30769230769|
|Electronics|           30375.0|
|   Clothing|         68415.625|
|  Furniture|           62340.0|
+-----------+------------------+



In [38]:
df.groupBy("region") \
  .agg(
      sum("total_sales").alias("sales")
  ) \
  .filter(col("sales") > 50000) \
  .show()

+-------+-------+
| region|  sales|
+-------+-------+
|Unknown| 396250|
|  South|1226800|
|Central|1747850|
|   East|1609550|
|   West| 697800|
|  North|1100350|
+-------+-------+



In [39]:
df.groupBy("region").count().show()

+-------+-----+
| region|count|
+-------+-----+
|Unknown|   15|
|  South|   16|
|Central|   16|
|   East|   17|
|   West|   15|
|  North|   18|
+-------+-----+



In [40]:
result = (
    df.dropDuplicates()
      .fillna({"age": 0, "region": "Unknown"})
      .groupBy("category")
      .agg(
          count("*").alias("orders"),
          sum("total_sales").alias("sales"),
          avg("total_sales").alias("avg_sales")
      )
)

result.show()

+-----------+------+-------+------------------+
|   category|orders|  sales|         avg_sales|
+-----------+------+-------+------------------+
|     Sports|    19|1236300| 65068.42105263158|
|    Grocery|    26|2714850|104417.30769230769|
|Electronics|    16| 486000|           30375.0|
|   Clothing|    16|1094650|         68415.625|
|  Furniture|    20|1246800|           62340.0|
+-----------+------+-------+------------------+

